In [2]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu datasets fastapi uvicorn pyngrok nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.4 MB/s eta 0:00:00:00:0100:01


In [3]:
import os
import re
import json
import threading
import numpy as np
import torch
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

import uvicorn
from pyngrok import ngrok

In [4]:
dataset = load_dataset("fancyzhx/ag_news")

print(dataset)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [5]:
NUM_DOCUMENTS = 5000

documents = dataset["train"].select(
    range(NUM_DOCUMENTS)
)

texts = documents["text"]
labels = documents["label"]

print("Number of documents:", len(texts))
print("Example:")
print(texts[0])

Number of documents: 5000
Example:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [6]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [7]:
embeddings = embedding_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Embedding shape: (5000, 384)


In [8]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print("FAISS index created.")
print("Indexed documents:", index.ntotal)

FAISS index created.
Indexed documents: 5000


In [9]:
def retrieve_evidence(query, k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1
    ):

        idx = int(idx)

        results.append({
            "rank": rank,
            "document_id": idx,
            "similarity": float(score),
            "category": int(labels[idx]),
            "text": texts[idx]
        })

    return results


test_results = retrieve_evidence(
    "artificial intelligence technology",
    k=5
)

for item in test_results:

    print("=" * 80)

    print("Evidence:", item["rank"])
    print("Document:", item["document_id"])
    print("Similarity:", item["similarity"])
    print("Category:", item["category"])
    print("Text:", item["text"][:500])

Evidence: 1
Document: 553
Similarity: 0.5137258768081665
Category: 3
Text: NASA Develops Robust Artificial Intelligence for Planetary Rovers NASA is planning to add a strong dose of artificial intelligence (AI) to planetary rovers to make them much more self-reliant, capable of making basic decisions during a mission. Scientists are developing very complex AI software that enables a higher level of robotic intelligence.
Evidence: 2
Document: 741
Similarity: 0.43756720423698425
Category: 2
Text: Computers with multiple personalities The jury's still out on whether a computer can ever truly be intelligent, but there's no question that it can have multiple personalities. It's just a matter of software.
Evidence: 3
Document: 1660
Similarity: 0.3827645480632782
Category: 3
Text: Indian researcher designs revolutionary low cost PC for the poor Aug 17 - Professor Raj Reddy, an Indian researcher in artificial intelligence and a professor at Carnegie Mellon University. Has designed a wirelessly

In [10]:
MODEL_NAME = "mistralai/Mistral-Nemo-Instruct-2407"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

model.eval()

print("Mistral Nemo loaded successfully.")

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] The tokenizer you are loading from 'mistralai/Mistral-Nemo-Instruct-2407' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral Nemo loaded successfully.


In [11]:
def build_intelligence_prompt(topic, evidence):

    evidence_text = ""

    for item in evidence:

        evidence_text += f"""
[EVIDENCE {item['rank']}]

Document ID:
{item['document_id']}

Similarity:
{item['similarity']:.4f}

Category:
{item['category']}

Text:
{item['text']}

----------------------------------------
"""


    prompt = f"""
You are InsightLens, an AI Intelligence Analyst.

Your job is to analyze the user's topic using ONLY
the retrieved evidence provided below.

USER TOPIC:
{topic}

RETRIEVED EVIDENCE:
{evidence_text}

STRICT RULES:

1. Do not invent facts.
2. Do not use outside knowledge.
3. Use only the retrieved evidence.
4. Clearly distinguish evidence from interpretation.
5. Mention evidence numbers for important findings.
6. If the evidence is insufficient, explicitly say so.
7. Similarity scores indicate semantic relevance,
   not factual truth.
8. Do not claim that an article proves something
   unless the evidence actually supports it.

Return the answer using EXACTLY this structure:

EXECUTIVE SUMMARY:
Write a concise 2-3 sentence summary.

KEY FINDINGS:
- Finding 1 [Evidence X]
- Finding 2 [Evidence X]
- Finding 3 [Evidence X]

SUPPORTING SIGNALS:
- Signal 1 [Evidence X]
- Signal 2 [Evidence X]

CONFLICTING OR DIFFERENT SIGNALS:
- Signal 1 [Evidence X]
- Signal 2 [Evidence X]

RISKS / OPPORTUNITIES:
- Risk or opportunity 1
- Risk or opportunity 2

CONFIDENCE:
High / Medium / Low

CONFIDENCE REASON:
Explain why this confidence level was selected.

EVIDENCE USED:
- Evidence 1: Document ID X
- Evidence 2: Document ID X
- Evidence 3: Document ID X
"""

    return prompt

In [12]:
def generate_intelligence_brief(topic, evidence):

    prompt = build_intelligence_prompt(
        topic,
        evidence
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=700,
            temperature=0.2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [13]:
def analyze_topic(topic):

    if not topic or not topic.strip():

        return {
            "status": "error",
            "message": "Please enter a topic."
        }

    topic = topic.strip()

    evidence = retrieve_evidence(
        topic,
        k=5
    )

    brief = generate_intelligence_brief(
        topic,
        evidence
    )

    return {
        "status": "success",
        "topic": topic,
        "brief": brief,
        "evidence": evidence
    }


result = analyze_topic(
    "Artificial intelligence is transforming the technology industry."
)

print(result["brief"])

print("\n\nRETRIEVED EVIDENCE:")

for item in result["evidence"]:

    print("=" * 80)

    print(
        f"Evidence {item['rank']} | "
        f"Similarity: {item['similarity']:.3f}"
    )

    print(item["text"][:500])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


- Evidence 4: Document ID X
- Evidence 5: Document ID X

---

EXECUTIVE SUMMARY:
Artificial intelligence is transforming the technology industry, as seen in NASA's development of advanced AI for planetary rovers (Evidence 3). However, the impact of AI on employment and growth in the tech industry is mixed, with some signs of optimism (Evidence 1) and others indicating potential job exports (Evidence 5).

KEY FINDINGS:
- AI is being developed for advanced applications, such as NASA's planetary rovers (Evidence 3).
- There is optimism in the tech industry, with unemployment dropping (Evidence 1).
- There are predictions of growth in IT job exports (Evidence 5).

SUPPORTING SIGNALS:
- Optimism in the tech industry is indicated by dropping unemployment (Evidence 1).
- Growth in IT job exports is predicted (Evidence 5).

CONFLICTING OR DIFFERENT SIGNALS:
- While there is optimism in the tech industry, there are also predictions of job exports (Evidence 5).

RISKS / OPPORTUNITIES:
- Opportun

In [14]:
def analyze_topic(topic):

    if not topic or not topic.strip():

        return {
            "status": "error",
            "message": "Please enter a topic."
        }

    topic = topic.strip()

    evidence = retrieve_evidence(
        topic,
        k=5
    )

    brief = generate_intelligence_brief(
        topic,
        evidence
    )

    return {
        "status": "success",
        "topic": topic,
        "brief": brief,
        "evidence": evidence
    }


result = analyze_topic(
    "Artificial intelligence is transforming the technology industry."
)

print(result["brief"])

print("\n\nRETRIEVED EVIDENCE:")

for item in result["evidence"]:

    print("=" * 80)

    print(
        f"Evidence {item['rank']} | "
        f"Similarity: {item['similarity']:.3f}"
    )

    print(item["text"][:500])

- Evidence 4: Document ID X
- Evidence 5: Document ID X

---

EXECUTIVE SUMMARY:
Artificial intelligence is transforming the technology industry, as seen in NASA's development of advanced AI for planetary rovers (Evidence 3). However, the impact of AI on employment and growth in the tech industry is mixed, with some signs of optimism (Evidence 1) and others indicating potential job exports (Evidence 5).

KEY FINDINGS:
- AI is being developed for advanced applications, such as NASA's planetary rovers (Evidence 3).
- There is optimism in the tech industry, with unemployment dropping (Evidence 1).
- There are predictions of growth in IT job exports (Evidence 5).

SUPPORTING SIGNALS:
- Optimism in the tech industry is indicated by dropping unemployment (Evidence 1).
- Growth in IT job exports is predicted (Evidence 5).

CONFLICTING OR DIFFERENT SIGNALS:
- While there is optimism in the tech industry, there are also predictions of job exports (Evidence 5).

RISKS / OPPORTUNITIES:
- Opportun

In [15]:
app = FastAPI(
    title="InsightLens API",
    description="Evidence-Backed AI Intelligence System",
    version="1.0.0"
)


API_KEY = "xxxxxxx"


class TopicRequest(BaseModel):

    topic: str


@app.get("/")
def root():

    return {
        "project": "InsightLens",
        "status": "online",
        "message": "Evidence-Backed AI Intelligence API"
    }


@app.get("/health")
def health():

    return {
        "status": "healthy"
    }


@app.post("/intelligence")
def intelligence(
    request: TopicRequest,
    authorization: str = Header(None)
):

    if authorization != f"Bearer {API_KEY}":

        raise HTTPException(
            status_code=401,
            detail="Invalid API key"
        )


    if not request.topic.strip():

        raise HTTPException(
            status_code=400,
            detail="Topic cannot be empty"
        )


    result = analyze_topic(
        request.topic
    )

    return result

In [16]:
def run_api():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

print("FastAPI server started on port 8000.")

FastAPI server started on port 8000.


NGROK_AUTH_TOKEN = "xxxxxxxxxxxxx"

ngrok.set_auth_token(
    NGROK_AUTH_TOKEN
)

public_url = ngrok.connect(
    8000,
    "http"
)

print("Public API URL:")
print(public_url)

In [18]:
import requests
import json

API_URL = public_url.public_url.rstrip("/")

response = requests.post(
    API_URL + "/intelligence",
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "topic": "Artificial intelligence is transforming the technology industry."
    },
    timeout=300
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

INFO:     34.136.88.159:0 - "POST /intelligence HTTP/1.1" 200 OK
Status: 200
{
  "status": "success",
  "topic": "Artificial intelligence is transforming the technology industry.",
  "brief": "- Evidence 4: Document ID X\n- Evidence 5: Document ID X\n\n---\n\nEXECUTIVE SUMMARY:\nArtificial intelligence is transforming the technology industry, as seen in NASA's development of advanced AI for planetary rovers (Evidence 3). However, the impact of AI on employment and growth in the tech industry is mixed, with some signs of optimism (Evidence 1) and others indicating potential job exports (Evidence 5).\n\nKEY FINDINGS:\n- AI is being developed for advanced applications, such as NASA's planetary rovers (Evidence 3).\n- There is optimism in the tech industry, with unemployment dropping (Evidence 1).\n- There are predictions of growth in IT job exports (Evidence 5).\n\nSUPPORTING SIGNALS:\n- Optimism in the tech industry is indicated by dropping unemployment (Evidence 1).\n- Growth in IT job 